# 32 — Chimera States Under FIM

**UNPRECEDENTED:** Chimera states (coexisting synchronised and desynchronised
populations) are a major topic in coupled oscillator theory since Kuramoto &
Battogtokh (2002). Nobody has tested what happens to chimeras under
self-observation (FIM).

**Hypotheses:**
- A: FIM destroys chimeras (forces global sync) — consistent with NB29
- B: FIM creates NEW types of chimeras (self-referential chimeras)
- C: There's a λ window where chimeras are STABILISED by FIM

## Setup
Non-local coupling (Kuramoto-Battogtokh): ring of N oscillators with
exponentially decaying coupling. This topology is known to support chimeras.

In [ ]:
import json
from pathlib import Path

import numpy as np


def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity


def build_nonlocal_ring(N, kappa=4.0):
    """Non-local coupling on a ring: K[i,j] = exp(-kappa * |i-j|/N * 2pi)."""
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i != j:
                dist = min(abs(i - j), N - abs(i - j))  # ring distance
                K[i, j] = np.exp(-kappa * dist / N)
    return K


def chimera_order_params(theta, N):
    """Local order parameter: R_i = |mean(exp(i*theta_j))| for j near i.
    Returns: global R, local R array, chimera metric chi."""
    half = N // 4  # window size
    R_local = np.zeros(N)
    for i in range(N):
        # Neighbours on ring
        indices = [(i + k) % N for k in range(-half, half + 1)]
        R_local[i] = np.abs(np.mean(np.exp(1j * theta[indices])))

    R_global = np.abs(np.mean(np.exp(1j * theta)))
    # Chimera metric: std of local R (high = chimera, low = uniform)
    chi = float(np.std(R_local))
    return float(R_global), R_local, chi


def simulate_chimera(N, K, omega, fim_lambda, K_scale=1.0, dt=0.01, T=200.0, noise=0.01, seed=42):
    """Simulate and detect chimera states."""
    rng = np.random.default_rng(seed)
    # Chimera-favouring IC: half aligned, half random
    theta = np.zeros(N)
    theta[: N // 2] = 0.0 + 0.01 * rng.normal(size=N // 2)  # coherent
    theta[N // 2 :] = rng.uniform(0, 2 * np.pi, N - N // 2)  # incoherent

    K_eff = K * K_scale
    n_steps = int(T / dt)

    # Collect metrics in last 25%
    R_global_tail = []
    chi_tail = []
    R_local_snapshot = None

    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K_eff * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if fim_lambda > 0:
            dphi += fim_lambda * fim_gradient_all(theta)
        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)

        if s >= n_steps * 3 // 4:
            R_g, R_l, chi = chimera_order_params(theta, N)
            R_global_tail.append(R_g)
            chi_tail.append(chi)
            if s == n_steps - 1:
                R_local_snapshot = R_l.copy()

    return {
        "R_global": float(np.mean(R_global_tail)),
        "chi_mean": float(np.mean(chi_tail)),  # chimera metric
        "chi_std": float(np.std(chi_tail)),
        "R_local_final": R_local_snapshot,
    }


N = 64
K_ring = build_nonlocal_ring(N, kappa=4.0)
# Heterogeneous frequencies (needed for chimeras)
rng = np.random.default_rng(42)
omega = rng.standard_normal(N) * 0.5

print(f"N={N}, ring coupling kappa=4.0")
print(f"omega spread: [{omega.min():.2f}, {omega.max():.2f}]")

In [ ]:
# Sweep K_scale and λ — detect chimeras
K_scales = [0.5, 1.0, 2.0, 3.0, 5.0]
lambdas = [0, 0.1, 0.5, 1.0, 2.0, 5.0]

print(f"{'K':>6} {'λ':>6} {'R':>8} {'χ (chimera)':>12} {'Type':>12}")
chimera_data = []

for K_scale in K_scales:
    for lam in lambdas:
        # Average over 3 seeds with chimera IC
        Rs, chis = [], []
        for seed in [42, 123, 456]:
            res = simulate_chimera(N, K_ring, omega, lam, K_scale, seed=seed)
            Rs.append(res["R_global"])
            chis.append(res["chi_mean"])

        R = np.mean(Rs)
        chi = np.mean(chis)

        # Classify:
        # χ > 0.15 AND R < 0.8 → CHIMERA
        # χ < 0.05 AND R > 0.8 → FULL SYNC
        # χ < 0.05 AND R < 0.3 → DESYNC
        if chi > 0.15 and R < 0.8:
            state = "CHIMERA"
        elif R > 0.8 and chi < 0.1:
            state = "SYNC"
        elif R < 0.3 and chi < 0.1:
            state = "DESYNC"
        elif chi > 0.1:
            state = "PARTIAL-CH"
        else:
            state = "PARTIAL"

        chimera_data.append(
            {"K": K_scale, "lambda": lam, "R": round(R, 4), "chi": round(chi, 4), "type": state}
        )
        print(f"{K_scale:6.1f} {lam:6.1f} {R:8.4f} {chi:12.4f} {state:>12}")

# Summary
print("\n=== CHIMERA CENSUS ===")
for state in ["CHIMERA", "PARTIAL-CH", "SYNC", "PARTIAL", "DESYNC"]:
    count = sum(1 for d in chimera_data if d["type"] == state)
    if count > 0:
        print(f"{state}: {count} points")

# Key question: does FIM destroy chimeras?
print("\n=== FIM EFFECT ON CHIMERAS ===")
for K_scale in K_scales:
    chi_no_fim = [d["chi"] for d in chimera_data if d["K"] == K_scale and d["lambda"] == 0][0]
    chi_fim5 = [d["chi"] for d in chimera_data if d["K"] == K_scale and d["lambda"] == 5.0][0]
    if chi_no_fim > 0.1:
        change = (
            "DESTROYED"
            if chi_fim5 < 0.05
            else ("REDUCED" if chi_fim5 < chi_no_fim else "PRESERVED")
        )
        print(f"K={K_scale}: χ(λ=0)={chi_no_fim:.4f} → χ(λ=5)={chi_fim5:.4f} — chimera {change}")

In [ ]:
# Save
output = {
    "experiment": "Chimera states under FIM (strange loop)",
    "N": N,
    "kappa": 4.0,
    "data": chimera_data,
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/chimera_states_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")